In [116]:
from datasets import load_dataset, DatasetDict, load_from_disk

In [117]:
data_dir ="/home/jovyan/Works/MSc_Thesis/mh_function_calling/dataset_generation/fc_training_data"
dataset = load_dataset("json",data_dir= data_dir)

Generating train split: 5099 examples [00:00, 39202.84 examples/s]


In [118]:
# See the unique query types
dataset["train"].unique("type")

['symptom', 'behavioral', 'metaphorical', 'explicit', 'implicit']

In [119]:
# a sample
dataset = dataset.rename_column("numdays","num_days")

In [120]:
dataset["train"][1]

{'query': "Show me my glucose trends for the past two weeks—I started a new workout routine and want to see if it's making a difference.",
 'type': 'behavioral',
 'thinking': "The user explicitly requests glucose data to correlate with a behavioral change (new workout routine). They specify 'past two weeks', so I'll retrieve 14 days of data to match their stated timeframe and assess impact.",
 'num_days': 14,
 'function': 'get_blood_glucose_data'}

In [121]:
import random

def add_patient_ids(dataset):
    """Add random patient IDs to the dataset"""
    
    # Get total number of samples
    num_samples = len(dataset["train"])
    
    # Generate unique patient IDs
    patient_ids = [f"P_{i+1:03d}" for i in range(num_samples)]
    
    # Shuffle for randomness
    random.shuffle(patient_ids)
    
    # Add to dataset
    dataset = dataset["train"].add_column("patient_id", patient_ids)
    
    return dataset

In [122]:

# Add patient IDs
dataset = add_patient_ids(dataset)

In [123]:
dataset[1]

{'query': "Show me my glucose trends for the past two weeks—I started a new workout routine and want to see if it's making a difference.",
 'type': 'behavioral',
 'thinking': "The user explicitly requests glucose data to correlate with a behavioral change (new workout routine). They specify 'past two weeks', so I'll retrieve 14 days of data to match their stated timeframe and assess impact.",
 'num_days': 14,
 'function': 'get_blood_glucose_data',
 'patient_id': 'P_3599'}

In [124]:
def create_assistant_response(example):
    """Create properly formatted assistant response"""
    
    # Clean the thinking text (remove any existing tags)
    thinking = example['thinking'].replace('<think>', '').replace('</think>', '').strip()
    
    # Create the formatted response
    response = f"""<think>
{thinking}
</think>

<function>
{{
  "name": "{example['function']}",
  "parameters": {{
    "patient_id": "{example['patient_id']}",
    "num_days": {example['num_days']}
  }}
}}
</function>"""
    
    return {"assistant_response": response}

In [125]:
# Add assistant response to all samples
dataset = dataset.map(create_assistant_response)
print("✅ Added assistant_response column")

# Also rename query column for consistency
dataset = dataset.rename_column("query", "patient_query")
print("✅ Renamed query to patient_query")

Map: 100%|██████████| 5099/5099 [00:00<00:00, 14750.39 examples/s]

✅ Added assistant_response column
✅ Renamed query to patient_query


In [126]:
dataset[0]

{'patient_query': "I've been feeling really shaky and lightheaded after meals lately. Could my blood sugar be spiking?",
 'type': 'symptom',
 'thinking': "The user is describing physical symptoms (shakiness, lightheadedness) that suggest concern about post-meal blood glucose spikes. Since they mention 'lately' but don't specify a timeframe, I'll default to 7 days of data to identify patterns after meals.",
 'num_days': 7,
 'function': 'get_blood_glucose_data',
 'patient_id': 'P_048',
 'assistant_response': '<think>\nThe user is describing physical symptoms (shakiness, lightheadedness) that suggest concern about post-meal blood glucose spikes. Since they mention \'lately\' but don\'t specify a timeframe, I\'ll default to 7 days of data to identify patterns after meals.\n</think>\n\n<function>\n{\n  "name": "get_blood_glucose_data",\n  "parameters": {\n    "patient_id": "P_048",\n    "num_days": 7\n  }\n}\n</function>'}

In [127]:
from transformers import AutoTokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [128]:
def prepare_dataset_with_text_column(dataset):
   
    
    # Data prompt
    data_prompt = """You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: {}
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        patient_ids = examples["patient_id"]
        queries = examples["patient_query"]
        responses = examples["assistant_response"]
        texts = []
        
        for patient_id, query, response in zip(patient_ids, queries, responses):
            text = data_prompt.format(patient_id, query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [129]:
dataset = prepare_dataset_with_text_column(dataset)

Map: 100%|██████████| 5099/5099 [00:00<00:00, 92246.32 examples/s]


In [130]:
print(dataset[502]["text"])

You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: P_3410
Pateint Query: Can you check if my BMR has changed over the past two weeks?

### Response:
<think>
The user is directly asking to analyze potential changes in their basal metabolic rate. I will retrieve BMR data for the past 14 days using get_bmr_data to track any variations.
</think>

<function>
{
  "name": "get_bmr_data",
  "parameters": {
    "patient_id": "P_3410",
    "num_days": 14
  }
}
</function><|eot_id|>


In [131]:
dataset = dataset.shuffle(seed=42)

In [132]:
dataset[3]

{'patient_query': 'I’ve been taking evening strolls—show the past 11 days of steps.',
 'type': 'explicit',
 'thinking': 'Evening habit check benefits from a week plus a few days; 11 days fits well.',
 'num_days': 11,
 'function': 'get_steps_data',
 'patient_id': 'P_4620',
 'assistant_response': '<think>\nEvening habit check benefits from a week plus a few days; 11 days fits well.\n</think>\n\n<function>\n{\n  "name": "get_steps_data",\n  "parameters": {\n    "patient_id": "P_4620",\n    "num_days": 11\n  }\n}\n</function>',
 'text': 'You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.\n\n### Input:\nPatient ID: P_4620\nPateint Query: I’ve been taking evening strolls—show the past 11 days of steps.\n\n### Respons

In [133]:
dataset = dataset.train_test_split(train_size=0.75, seed=42)

In [134]:
dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 3824
    })
    test: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 1275
    })
})

In [135]:
test_and_val_split = dataset["test"].train_test_split(train_size=0.60, seed=42)

In [136]:
test_and_val_split

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 765
    })
    test: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 510
    })
})

In [137]:
dataset["train"] = dataset["train"]
dataset["test"] = test_and_val_split["train"]
dataset["validation"] = test_and_val_split["test"]

In [138]:
dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 3824
    })
    test: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 765
    })
    validation: Dataset({
        features: ['patient_query', 'type', 'thinking', 'num_days', 'function', 'patient_id', 'assistant_response', 'text'],
        num_rows: 510
    })
})

In [139]:
len(dataset["train"].unique("function"))

Flattening the indices: 100%|██████████| 3824/3824 [00:00<00:00, 23001.41 examples/s]


20

In [140]:
len(dataset["test"].unique("function"))

Flattening the indices: 100%|██████████| 765/765 [00:00<00:00, 28528.62 examples/s]


20

In [141]:
len(dataset["validation"].unique("function"))

Flattening the indices: 100%|██████████| 510/510 [00:00<00:00, 30678.59 examples/s]


20

In [142]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset.save_to_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf-v3")

Saving the dataset (1/1 shards): 100%|██████████| 510/510 [00:00<00:00, 38203.58 examples/s]


In [143]:
dataset = load_from_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf-v3")

In [161]:
print(dataset["test"][364]["text"])

You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: P_954
Pateint Query: Can you check my gym attendance for the last 15 days?

### Response:
<think>
The user explicitly requests an analysis of gym attendance over a specific period. This requires retrieving session data for frequency. ‘Last 15 days’ clearly maps to numdays of 15.
</think>

<function>
{
  "name": "get_exercise_session_data",
  "parameters": {
    "patient_id": "P_954",
    "num_days": 15
  }
}
</function><|eot_id|>
